# Assignment 2: observation inspector

This notebook builds observations with **your** `submission/observation.py` on the
eight public probe scenes, so you can see exactly what the model would receive:
the image and the text with it. It loads your file through the same loader the
probe and the structural gates use, and applies your `submission/agent.yaml`.
What you see here is what `probe --submission` sends.

It never runs MetaDrive or a model. The scenes were recorded once, each with its
camera frame and the scene DriveBench measured with it, in
`fixtures/vla/probes/catalog-20261002-v5/`. To see what a model answers, probe your
observation and read the result in `probe-analysis.ipynb`.

Your `observation.py` has three tasks, each at a named `TODO` line: the
instructions (section 2), the facts the text gives (section 3), and the image
(section 5). Each of those sections ends with the structural gates' own
observation checks, printed as `PASS` or `NOT YET`. They tell you whether your
observation can be sent, never how well a model answers it: the probe measures
that.

The notebook is never submitted and never graded. Rerun it from the top after
every change; each run imports your files afresh.

Start it from the repository root:

```bash
uv run --extra notebooks jupyter lab course/assignment-2/observation-inspector.ipynb
```

In [ ]:
import difflib
import time
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from metadrive_starter.config import load_config
from metadrive_starter.gates import CONTROL_LOOP_UNBLOCKED, VALID_OBSERVATIONS, load_gate_plan
from metadrive_starter.recorded_probes import (
    INTERPRETATION_CATALOG,
    load_recorded_probes,
    observation_request,
)
from metadrive_starter.submission import (
    PROJECT_ROOT,
    apply_submission_overlay,
    load_observation,
)
from metadrive_starter.vla.observation import (
    OUTPUT_CONTRACT,
    ObservationError,
    checked_observation,
    describe_missing_contract,
    missing_contract_lines,
)
from metadrive_starter.vla.prompting import DefaultObservationBuilder

SUBMISSION = PROJECT_ROOT / "submission"
# The configuration you probe with: its vla and controller values shape every
# request. Set it to the --config of your probe command. The brief's llama-server
# probe configurations all send what this one does, but for the interpretation
# profile, which section 6 builds.
BASE_CONFIG = PROJECT_ROOT / "configs" / "vla-probe-smolvlm-llama.yaml"
# The structural gates' configuration: they build every observation within its
# request timeout.
GATE_CONFIG = PROJECT_ROOT / "configs" / "demo-vla-fixture.yaml"
# The models your observation is marked on: the gates build every request for each.
GATE_MODELS = load_gate_plan(PROJECT_ROOT / "configs" / "gates-assignment-2.yaml").observation_models
config = apply_submission_overlay(load_config(BASE_CONFIG), SUBMISSION)
ObservationBuilder = load_observation(SUBMISSION)
probes = {probe.scenario_id: probe for probe in load_recorded_probes()}
# Scene interpretation's profile, and the public scenes that hold what it marks.
INTERPRETATION_CONFIG = PROJECT_ROOT / "configs" / "vla-probe-qwen3-vl-2b-llama-interpretation.yaml"
interpretation_config = apply_submission_overlay(load_config(INTERPRETATION_CONFIG), SUBMISSION)
interpretation_probes = {
    probe.scenario_id: probe for probe in load_recorded_probes(INTERPRETATION_CATALOG)
}
gate_timeout_s = load_config(GATE_CONFIG).vla.request_timeout_s


def build(probe, run_config=None):
    """Your observation of one recorded scene, checked as every run checks it."""
    run_config = config if run_config is None else run_config
    request = observation_request(probe, run_config)
    builder = ObservationBuilder(run_config.observation)
    return checked_observation(builder.build(request), request)


def original(probe):
    """DriveBench's original observation of the same request."""
    return DefaultObservationBuilder().build(observation_request(probe, config))


def image(frame):
    return np.frombuffer(frame.rgb_bytes, dtype=np.uint8).reshape(frame.height, frame.width, 3)


def describe(observation):
    prompt = observation.prompt
    print(
        f"{observation.frame.width}x{observation.frame.height} image; "
        f"{len(prompt)} characters, {len(prompt.split())} words, "
        f"{len(prompt.splitlines())} lines of text"
    )
    missing = missing_contract_lines(prompt, OUTPUT_CONTRACT)
    if not missing:
        print("every output-contract line is present")
    for line in missing:
        print(f"missing output-contract line: {line}")


def observation_checks():
    """The structural gates' observation checks on every scene: (gate, passed, detail).

    One builder builds every scene's observation, as in a run, and each build is
    timed and checked as the gates check it.
    """
    builder = ObservationBuilder(config.observation)
    problems, without_contract, slowest_s = [], [], 0.0
    requests = [
        (f"{scene_id} for {model}", replace(observation_request(probe, config), model_id=model))
        for scene_id, probe in probes.items()
        for model in GATE_MODELS
    ]
    for label, request in requests:
        started_s = time.perf_counter()
        try:
            observation = builder.build(request)
        except Exception as exc:
            problems.append(f"{label}: build(request) raised {type(exc).__name__}: {exc}")
            continue
        finally:
            slowest_s = max(slowest_s, time.perf_counter() - started_s)
        try:
            checked_observation(observation, request)
        except ObservationError as exc:
            problems.append(f"{label}: {exc}")
            continue
        missing = missing_contract_lines(observation.prompt, request.contract)
        if missing:
            without_contract.append(missing)
    if problems:
        valid = (
            False,
            f"{len(problems)} of {len(requests)} observations could not be sent; "
            f"the first: {problems[0]}",
        )
    elif without_contract:
        # Reported, never failed: the output contract is a recommendation.
        valid = (
            True,
            f"{len(requests)} observations built, each with a valid frame; in "
            f"{len(without_contract)} of them the prompt "
            f"{describe_missing_contract(without_contract[0])}",
        )
    else:
        valid = (
            True,
            f"{len(requests)} observations built, each with a valid frame and the output contract",
        )
    if slowest_s <= gate_timeout_s:
        unblocked = (
            True,
            f"every observation was built within the gates' {gate_timeout_s:g} s request "
            f"timeout, the slowest in {1000.0 * slowest_s:.2f} ms",
        )
    else:
        unblocked = (
            False,
            f"the slowest observation took {1000.0 * slowest_s:.2f} ms to build, longer "
            f"than the gates' {gate_timeout_s:g} s request timeout",
        )
    return [(VALID_OBSERVATIONS, *valid), (CONTROL_LOOP_UNBLOCKED, *unblocked)]


def check():
    """Print the gates' observation checks of the eight scenes, each as PASS or NOT YET."""
    print(
        "The structural gates' checks of your observation on the eight scenes, "
        f"for {', '.join(GATE_MODELS)}:"
    )
    for gate, passed, detail in observation_checks():
        print(f"{'PASS   ' if passed else 'NOT YET'}  {gate}: {detail}")


print(
    f"{len(probes)} recorded scenes; requests built from "
    f"{BASE_CONFIG.relative_to(PROJECT_ROOT)} with your agent.yaml"
)
print("observation settings:", config.observation)
print("prompt policy:", config.vla.prompt_policy)

## The eight scenes

What each probe scene contains, what the probe accepts as an answer, and what
DriveBench measured in it. Assignment 2's requests withhold the measured objects
and traffic lights (`request.withheld` names them), so your observation's text
can describe the lanes beside the car but no object: the objects listed here are
what the image has to show the model, and what your arbiter is given.

In [ ]:
def lane(available):
    return {True: "available", False: "unavailable", None: "unknown"}[available]


def side(lateral_m):
    return f"{abs(lateral_m):.1f} m {'left' if lateral_m >= 0 else 'right'}"


for probe in probes.values():
    scenario, scene = probe.scenario, probe.scene
    print(f"{scenario.scenario_id}: {scenario.description}")
    evaluation = scenario.evaluation
    if evaluation is not None:
        actions = ", ".join(action.value for action in evaluation.acceptable_actions)
        print(
            f"  the probe accepts {actions} at {evaluation.minimum_target_speed_mps:.1f} "
            f"to {evaluation.maximum_target_speed_mps:.1f} m/s"
        )
        for hazard in evaluation.required_hazards:
            print(
                f"  and requires a {hazard.hazard_type.value} hazard "
                f"at {hazard.relative_location.value}"
            )
    for tracked in scene.objects:
        path = "in the ego path" if tracked.in_path else "off the ego path"
        print(
            f"  measured: {tracked.kind}, {tracked.relative_position_m[0]:.1f} m ahead "
            f"and {side(tracked.relative_position_m[1])}, {path}"
        )
    for light in scene.traffic_lights:
        print(
            f"  measured: {light.state.value} signal head, "
            f"{light.relative_position_m[0]:.1f} m ahead and {side(light.relative_position_m[1])}"
        )
    print(
        f"  measured: left lane {lane(scene.left_lane_available)}, "
        f"right lane {lane(scene.right_lane_available)}"
    )
    print()

## 1. What the model receives

One scene's observation, exactly as your `observation.py` builds it: the image
first, then the text. Change `SCENE` to any of the ids above.

In [ ]:
SCENE = "stopped-vehicle"

observation = build(probes[SCENE])
plt.figure(figsize=(9, 5))
plt.imshow(image(observation.frame))
plt.title(f"{SCENE}: the image your observation sends")
plt.axis("off")
plt.show()
describe(observation)
print()
print(observation.prompt)

- Which facts does the text give that the model could only guess from the image?
  Which does the image show that the text leaves out?
- The request withholds the stopped car, so only the image shows it. What in the
  image tells the model how far ahead it is, and in which lane?
- How much of the text is instructions, and how much is about this scene? The
  model reads all of it on every request.
- Set `SCENE = "partial-occlusion"`. What does the image show of the car the
  truck hides? The text cannot name it.

## 2. Yours against the original

DriveBench's original observation, `build_vla_prompt`, is what a run without
`--submission` sends. The shipped `observation.py` reproduces it less one line,
which a `TODO` replaces. Here is what your file changes on every scene, and the
changed lines on `SCENE`.

**Task 1: the instructions.** At `# TODO: write the instructions your model
needs` in `submission/observation.py`, decide what the text asks of the model,
then rerun the notebook from the top. `request.model_id` names the model each
request goes to, so you may instruct SmolVLM and Qwen differently: this notebook
builds for the model `BASE_CONFIG` names, so set it to a Qwen configuration to
see what Qwen receives. The TODO inside that list, on answers the runtime
keeps, marks the line the shipped file leaves out of the original: the gates
cannot check it, but a probe of SmolVLM with structured output on can, as the
brief's model ladder describes, and SmolVLM's points count only the answers the
runtime accepts. The cell below ends with the gates' checks, which build
every scene for both models.

In [ ]:
for probe in probes.values():
    yours, theirs = build(probe), original(probe)
    changed_lines = [
        line
        for line in difflib.ndiff(theirs.prompt.splitlines(), yours.prompt.splitlines())
        if line.startswith(("- ", "+ "))
    ]
    same_image = yours.frame == theirs.frame
    print(
        f"{probe.scenario_id:>24}: {len(changed_lines)} text lines changed, "
        f"image {'unchanged' if same_image else 'changed'}"
    )

print()
for line in difflib.unified_diff(
    original(probes[SCENE]).prompt.splitlines(),
    build(probes[SCENE]).prompt.splitlines(),
    "original",
    "yours",
    lineterm="",
):
    print(line)

print()
check()

## 3. The values in `agent.yaml`

Your `observation.py` owns how the observation is assembled; `agent.yaml` owns
the values it reads: `vla.prompt_policy` and the `observation` toggles. Each
variant below changes one value, for this notebook only, and shows what your
observation of `SCENE` gains or loses. A toggle changes nothing unless your
`observation.py` reads it, as the shipped one does.

**Task 2: the facts.** At `# TODO: choose which measured facts the text
gives` in `submission/observation.py`, decide which measured facts the text
describes, and how precisely; a toggle below leaves one out without code. The
cell below ends with the gates' checks of your observation as your
`agent.yaml` sets it, not of the variants.

In [ ]:
def with_observation(**changes):
    return replace(config, observation=replace(config.observation, **changes))


def with_scene_fields(**changes):
    fields = replace(config.observation.scene_fields, **changes)
    return replace(config, observation=replace(config.observation, scene_fields=fields))


def with_policy(text):
    return replace(config, vla=replace(config.vla, prompt_policy=text))


variants = {
    "observation.driving_context: false": with_observation(driving_context=False),
    "observation.scene_context: false": with_observation(scene_context=False),
    "observation.scene_fields.lanes: false": with_scene_fields(lanes=False),
    "vla.prompt_policy: one sentence": with_policy("Stop for anything in your lane."),
}
yours = build(probes[SCENE])
for name, variant in variants.items():
    changed = build(probes[SCENE], variant)
    print(f"== {name}: {len(yours.prompt)} -> {len(changed.prompt)} characters")
    for line in difflib.unified_diff(
        yours.prompt.splitlines(), changed.prompt.splitlines(), lineterm="", n=0
    ):
        if not line.startswith(("---", "+++", "@@")):
            print(line)
    print()

check()

To keep a variant, write its value in `submission/agent.yaml` and rerun the
notebook from the top, for example:

```yaml
observation:
  scene_fields:
    lanes: false
```

- What does `observation.scene_fields.lanes: false` take out of the text? Which
  probe scenes need that fact for the right action, since the image cannot tell
  a lane you may take from the oncoming carriageway?
- If your observation drops a fact, which scenes' probe scores should move, and
  in which direction? Write the prediction down before you probe both versions.

## 4. Every scene at once

Your observation's image for each scene, with the size of its text and how many
output-contract lines it lacks. A run warns about and counts every observation
without them, and sends it anyway.

In [ ]:
figure, axes = plt.subplots(2, 4, figsize=(16, 6))
for axis, probe in zip(axes.flat, probes.values()):
    observation = build(probe)
    missing = len(missing_contract_lines(observation.prompt, OUTPUT_CONTRACT))
    axis.imshow(image(observation.frame))
    axis.set_title(
        f"{probe.scenario_id}\n{len(observation.prompt)} characters, "
        f"{missing} contract lines missing",
        fontsize=9,
    )
for axis in axes.flat:
    axis.axis("off")
figure.tight_layout()
plt.show()

## 5. The image is part of the observation

**Task 3: the image.** At `# TODO: crop, scale, or draw on the image here` in
`submission/observation.py`, decide whether any fact belongs in the image. You
may crop, scale, or draw on the frame, provided you keep its `timestamp_s`. Two
cautions before you do:

- **The probe's traffic lights are enlarged.** They render at `visual_scale`
  1.75, so that their state is legible at 512x288. Real signals at this
  resolution would be harder to read, so a traffic-light result on the probe
  does not generalise.
- **Servers resize images themselves.** Detail you keep or draw may not survive
  the model server's own resizing.

What lower resolutions keep of the two smallest things the probe asks about,
then the gates' checks, which refuse an image that has been re-dated:

In [ ]:
def reduced(pixels, factor):
    """Every factor-th pixel each way, shown at full size: what that resolution keeps."""
    return pixels[::factor, ::factor].repeat(factor, axis=0).repeat(factor, axis=1)


# Rows and columns of each 512x288 frame.
regions = {
    "red-light-with-traffic": (slice(95, 145), slice(225, 375)),
    "partial-occlusion": (slice(100, 195), slice(195, 305)),
}
figure, axes = plt.subplots(2, 3, figsize=(14, 6))
for row, (scene_id, (rows, columns)) in zip(axes, regions.items()):
    pixels = image(probes[scene_id].frame)
    for axis, factor in zip(row, (1, 2, 4)):
        axis.imshow(reduced(pixels, factor)[rows, columns])
        width, height = pixels.shape[1] // factor, pixels.shape[0] // factor
        axis.set_title(f"{scene_id}, {width}x{height}", fontsize=9)
for axis in axes.flat:
    axis.axis("off")
figure.tight_layout()
plt.show()
print("traffic light visual_scale:", probes["red-light-with-traffic"].scenario.traffic_light.visual_scale)

print()
check()

- Would a crop that keeps only the road ahead lose anything a probe scene checks?
- The request does not report the hidden car, and the image shows a sliver of
  it at most. What can your observation do for the model here, and what is left
  to your arbiter, which still sees the car in the measured scene?

## 6. Pedestrians, cones, and barriers

The probe places only cars and trucks, so nothing above shows the pedestrians,
cones, and barriers that scene interpretation marks too. These eight
recorded scenes hold them, at several distances and lanes, and each is built here
as Qwen's interpretation profile sends it: the lanes that exist, but not whether
each is clear. Change `INTERPRETATION_SCENE` to any title below. What the image
keeps of a cone at 512x288 is what a model has to find it from.


In [ ]:
INTERPRETATION_SCENE = "public-single-cone-left-13m"

figure, axes = plt.subplots(2, 4, figsize=(16, 6))
for axis, (scene_id, probe) in zip(axes.flat, interpretation_probes.items()):
    axis.imshow(image(build(probe, interpretation_config).frame))
    axis.set_title(scene_id, fontsize=8)
for axis in axes.flat:
    axis.axis("off")
figure.tight_layout()
plt.show()

observation = build(interpretation_probes[INTERPRETATION_SCENE], interpretation_config)
describe(observation)
print()
print(observation.prompt)


## Next: what the model answers

This notebook shows what the model receives. To see what it answers, probe your
observation against a model server, into a new directory every time, as the
[brief](README.md#probing-your-observation) describes, and read the result with
`probe-analysis.ipynb`. The probe renders its scenes afresh with MetaDrive; they
are these scenes, up to a few dozen pixels that differ from render to render.